# Onboarding - HCP
Lucas Queirós

# Semana 1 -  DataLoader e Tokenização

## Configuração do ambiente

In [ ]:
import os

# Verificando versões das libs
from importlib.metadata import version
print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

torch version: 2.11.0
tiktoken version: 0.12.0


In [ ]:
import requests

# Importando corpus (Hamlet)
url = "https://www.gutenberg.org/cache/epub/1524/pg1524.txt"
response = requests.get(url)
corpus = response.text

# Retirando cabeçalho do Gutenberg
start_marker = "ACT I"
end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK HAMLET ***"

start = corpus.find(start_marker)
end = corpus.find(end_marker)
corpus = corpus[start:end].strip()

# Salvando o corpus para uso futuro
pasta = '../../data/'
arquivo = 'hamlet.txt'
path = os.path.join(pasta, arquivo)

# Salvando o arquivo
with open(path, 'w', encoding='utf-8') as f:
    f.write(corpus)

print(f"Corpus salvo em: {path}")

Corpus salvo em: ../../data/hamlet.txt


In [23]:
print(f"Numero de Caracteres: {len(corpus)}")
print(corpus[:300])

Numero de Caracteres: 184563
ACT I
 Scene I. Elsinore. A platform before the Castle
 Scene II. Elsinore. A room of state in the Castle
 Scene III. A room in Polonius’s house
 Scene IV. The platform
 Scene V. A more remote part of the Castle

 ACT II
 Scene I. A room in Polonius’s house
 Scene II. A room in the Castle



## Tokenizador rustico (regex)

In [24]:
import re

# tokenizer simples que converte texto em tokens com IDs utilizando regex e vice-versa
class SimpleTokenizer:

    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text) # qualquer um destes simbolos vira ponto de corte
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        # atribui o token de Unknow caso o token não esteja no vocabulario
        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)   # remove os espaços antes dos simbolos para tratar o split
        return text

In [25]:
# Criando vocabulario de tokens
def build_vocab(corpus):
    tokens = re.split(r'([,.:;?_!"()\']|--|\s)', corpus)
    tokens = [item.strip() for item in tokens if item.strip()]
    all_tokens = sorted(set(tokens))
    all_tokens.extend(["<|unk|>", "<|endoftext|>", "<|pad|>"])
    return {token: i for i, token in enumerate(all_tokens)}

# Passamos todo o corpus pela função e depois pelo tokenizador
vocab = build_vocab(corpus)
simple_tokenizer = SimpleTokenizer(vocab)

## DataLoader manual

In [26]:
import numpy as np

# DataLoader manual implementado com numpy
class SimpleDataLoader:
    def __init__(self, tokens, batch_size, seq_len, shuffle=True, pad_token=0):
        self.tokens = np.array(tokens, dtype=np.int32)
        self.batch_size = batch_size
        self.seq_len = seq_len
        self.shuffle = shuffle      # embaralha os chunks
        self.pad_token = pad_token  # id usado para token de padding

        chunk_size = seq_len + 1
        n_chunks = len(self.tokens) // chunk_size

        # Verifica se há tokens sobrando que não formam um chunk completo
        remainder = len(self.tokens) % chunk_size
        if remainder > 0:
            # Completa o chunk parcial com padding
            pad_size = chunk_size - remainder
            padding = np.full(pad_size, pad_token, dtype=np.int32)
            self.tokens = np.concatenate([self.tokens, padding])
            n_chunks += 1

        self.chunks = self.tokens.reshape(n_chunks, chunk_size)

    # itera sobre os batches
    def __iter__(self):
        indices = np.arange(len(self.chunks))

        if self.shuffle:
            np.random.shuffle(indices)

        for start in range(0, len(indices), self.batch_size):
            batch_idx = indices[start : start + self.batch_size]
            batch = self.chunks[batch_idx]

            # Se o último batch tiver menos que batch_size, completa com chunks de padding
            if len(batch) < self.batch_size:
                pad_chunks_needed = self.batch_size - len(batch)
                pad_chunk = np.full((pad_chunks_needed, self.seq_len + 1), self.pad_token, dtype=np.int32)
                batch = np.concatenate([batch, pad_chunk], axis=0)

            x = batch[:, :-1]
            y = batch[:, 1:]

            yield x, y

    def __len__(self):
        return int(np.ceil(len(self.chunks) / self.batch_size))

## Tokenizador GPT-2

In [27]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

## Comparação: Hamlet

In [28]:
test = corpus[:100]

# tiktoken
print("=" * 15 + "TIKTOKEN" + "=" * 15)
print(test)
integers1 = tokenizer.encode(test, allowed_special={"<|endoftext|>"})
print(integers1)
tokens1 = tokenizer.decode(integers1)
print(tokens1)

# tokenizador simples
print("=" * 15 + "SIMPLE TOKENIZER" + "=" * 15)
print(test)
integers3 = simple_tokenizer.encode(test)
print(integers3)
tokens3 = simple_tokenizer.decode(integers3)
print(tokens3)

===============TIKTOKEN===============
ACT I
 Scene I. Elsinore. A platform before the Castle
 Scene II. Elsinore. A room of state in the
[10659, 314, 201, 198, 28315, 314, 13, 2574, 31369, 382, 13, 317, 3859, 878, 262, 11312, 201, 198, 28315, 2873, 13, 2574, 31369, 382, 13, 317, 2119, 286, 1181, 287, 262]
ACT I
 Scene I. Elsinore. A platform before the Castle
 Scene II. Elsinore. A room of state in the
===============SIMPLE TOKENIZER===============
ACT I
 Scene I. Elsinore. A platform before the Castle
 Scene II. Elsinore. A room of state in the
[8, 364, 651, 364, 3, 210, 3, 7, 3740, 1202, 4748, 124, 651, 365, 3, 210, 3, 7, 4135, 3500, 4552, 2841, 4748]
ACT I Scene I. Elsinore. A platform before the Castle Scene II. Elsinore. A room of state in the


## Pipeline utilizando pytorch

In [29]:
# Repetiremos o processo feito acima porem utilizando as ferramentas do pytorch
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

class dataset_gpt2(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokeniza todo o texto de uma vez utilizando o tiktoken
        # O parametro allowed_special faz com que o tokenizer não acuse erro ao encontrar token desconhecidos
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Janela deslizante sobre os tokens
        # Cada iteração avança stride posições e recorta uma janela de max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]

            # converte cada janela em tensores e salva na lista de tensores criadas no construtor
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


# Método que cria todo o dataloader e gerencia a pipeline
def create_dataloader(txt, batch_size, max_length, stride,
                         shuffle=True, drop_last=True, num_workers=0):

    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = dataset_gpt2(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)

    return dataloader


In [30]:
# hiperparametros
BATCH_SIZE = 8
MAX_LENGTH = 128
STRIDE = 64

dataloader = create_dataloader(corpus, batch_size=BATCH_SIZE, max_length=MAX_LENGTH, stride=STRIDE)

In [31]:
for batch_idx, (input_ids, target_ids) in enumerate(dataloader):
    print(f"Batch {batch_idx}:")
    print(f"  Input shape: {input_ids.shape}")
    print(f"  Target shape: {target_ids.shape}")


Batch 0:
  Input shape: torch.Size([8, 128])
  Target shape: torch.Size([8, 128])
Batch 1:
  Input shape: torch.Size([8, 128])
  Target shape: torch.Size([8, 128])
Batch 2:
  Input shape: torch.Size([8, 128])
  Target shape: torch.Size([8, 128])
Batch 3:
  Input shape: torch.Size([8, 128])
  Target shape: torch.Size([8, 128])
Batch 4:
  Input shape: torch.Size([8, 128])
  Target shape: torch.Size([8, 128])
Batch 5:
  Input shape: torch.Size([8, 128])
  Target shape: torch.Size([8, 128])
Batch 6:
  Input shape: torch.Size([8, 128])
  Target shape: torch.Size([8, 128])
Batch 7:
  Input shape: torch.Size([8, 128])
  Target shape: torch.Size([8, 128])
Batch 8:
  Input shape: torch.Size([8, 128])
  Target shape: torch.Size([8, 128])
Batch 9:
  Input shape: torch.Size([8, 128])
  Target shape: torch.Size([8, 128])
Batch 10:
  Input shape: torch.Size([8, 128])
  Target shape: torch.Size([8, 128])
Batch 11:
  Input shape: torch.Size([8, 128])
  Target shape: torch.Size([8, 128])
Batch 12:
  In